# PS S6E6 - 027 Blend add026 RealMLP v5 check

Experiment: `exp_20260608_027_blend_add026_realmlp_v5_check`

Purpose:
- Add `026 RealMLP v5 as-is` to the current candidate pool.
- Check whether 026 adds complementary signal to `019 / 024 / 020 / 017 / 015 / 018`.
- Evaluate individual CV, pairwise OOF correlation, disagreement, error overlap, and safe blends.
- Save only the best safe blend OOF/pred `.npy` files for later use.

Important:
- `025` is intentionally excluded from the model pool because it was not promoted after Public LB dropped.
- LogReg/Ridge/ElasticNet rows are diagnostic only and must not be treated as final OOF unless a nested stacker is built.

In [1]:
# ============================================================
# 0. Imports / Config
# ============================================================

import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import spearmanr

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Ridge, ElasticNet

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260608_027_blend_add026_realmlp_v5_check"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

OUTDIR = Path("/kaggle/working") / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

MODEL_SPECS = [
    {"key": "014_blend_bias", "exp_id": "exp_20260605_014_bias_search_on_013_best_blend", "family": "Blend", "role": "old_blend_bias_reference", "oof": "oof_exp_20260605_014_bias_search_on_013_best_blend_proba.npy", "pred": "pred_exp_20260605_014_bias_search_on_013_best_blend_proba.npy", "cv": 0.9660085788215015, "public_lb": 0.96703},
    {"key": "015_realmlp_seed42", "exp_id": "exp_20260605_015_shared001_realmlp_as_is", "family": "RealMLP", "role": "stable_single_realmlp_backup", "oof": "oof_exp_20260605_015_shared001_realmlp_as_is_proba.npy", "pred": "pred_exp_20260605_015_shared001_realmlp_as_is_proba.npy", "cv": 0.9681693449222352, "public_lb": 0.96977},
    {"key": "017_realmlp_bias", "exp_id": "exp_20260606_017_bias_search_on_015_realmlp", "family": "RealMLP", "role": "previous_best_realmlp_bias_backup", "oof": "oof_exp_20260606_017_bias_search_on_015_realmlp_proba.npy", "pred": "pred_exp_20260606_017_bias_search_on_015_realmlp_proba.npy", "cv": 0.9683022653955233, "public_lb": 0.96985},
    {"key": "018_xgb_shared003", "exp_id": "exp_20260606_018_shared003_xgb_as_is", "family": "XGBoost", "role": "effective_blend_material", "oof": "oof_exp_20260606_018_shared003_xgb_as_is_proba.npy", "pred": "pred_exp_20260606_018_shared003_xgb_as_is_proba.npy", "cv": 0.965207986418628, "public_lb": 0.96578},
    {"key": "019_blend_best", "exp_id": "exp_20260607_019_blend_add018_xgb_check", "family": "Blend", "role": "public_confirmed_current_best", "oof": "oof_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy", "pred": "pred_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy", "cv": 0.968872315017554, "public_lb": 0.97000},
    {"key": "020_blend_bias", "exp_id": "exp_20260607_020_bias_search_on_019_best_blend", "family": "Blend", "role": "cv_trusted_attack_reference", "oof": "oof_exp_20260607_020_bias_search_on_019_best_blend_proba.npy", "pred": "pred_exp_20260607_020_bias_search_on_019_best_blend_proba.npy", "cv": 0.9692240443617589, "public_lb": 0.96969},
    {"key": "024_seed_ensemble_blend", "exp_id": "exp_20260608_024_seed_ensemble_and_blend_check", "family": "Blend", "role": "cv_trusted_slot", "oof": "oof_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy", "pred": "pred_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy", "cv": 0.9694803109983217, "public_lb": 0.96958},
    {"key": "026_realmlp_v5", "exp_id": "exp_20260608_026_realmlp_v5_as_is", "family": "RealMLP", "role": "realmlp_v5_single_model_candidate", "oof": "oof_exp_20260608_026_realmlp_v5_as_is_proba.npy", "pred": "pred_exp_20260608_026_realmlp_v5_as_is_proba.npy", "cv": 0.9690389777981231, "public_lb": 0.96979},
]

BLEND_SETS = {
    "A_019_only": ["019_blend_best"],
    "B_024_only": ["024_seed_ensemble_blend"],
    "C_020_only": ["020_blend_bias"],
    "D_026_only": ["026_realmlp_v5"],
    "E_017_only": ["017_realmlp_bias"],
    "F_019_026": ["019_blend_best", "026_realmlp_v5"],
    "G_024_026": ["024_seed_ensemble_blend", "026_realmlp_v5"],
    "H_020_026": ["020_blend_bias", "026_realmlp_v5"],
    "I_017_026": ["017_realmlp_bias", "026_realmlp_v5"],
    "J_015_026": ["015_realmlp_seed42", "026_realmlp_v5"],
    "K_019_024_026": ["019_blend_best", "024_seed_ensemble_blend", "026_realmlp_v5"],
    "L_019_020_026": ["019_blend_best", "020_blend_bias", "026_realmlp_v5"],
    "M_020_024_026": ["020_blend_bias", "024_seed_ensemble_blend", "026_realmlp_v5"],
    "N_019_020_024_026": ["019_blend_best", "020_blend_bias", "024_seed_ensemble_blend", "026_realmlp_v5"],
    "O_014_017_018_026": ["014_blend_bias", "017_realmlp_bias", "018_xgb_shared003", "026_realmlp_v5"],
    "P_014_018_019_026": ["014_blend_bias", "018_xgb_shared003", "019_blend_best", "026_realmlp_v5"],
    "Q_014_018_024_026": ["014_blend_bias", "018_xgb_shared003", "024_seed_ensemble_blend", "026_realmlp_v5"],
    "R_014_018_019_024_026": ["014_blend_bias", "018_xgb_shared003", "019_blend_best", "024_seed_ensemble_blend", "026_realmlp_v5"],
    "S_014_015_017_018_019_020_024_026": ["014_blend_bias", "015_realmlp_seed42", "017_realmlp_bias", "018_xgb_shared003", "019_blend_best", "020_blend_bias", "024_seed_ensemble_blend", "026_realmlp_v5"],
}

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)
print("NPY_DIR:", NPY_DIR)

EXP_ID: exp_20260608_027_blend_add026_realmlp_v5_check
OUTDIR: /kaggle/working/exp_20260608_027_blend_add026_realmlp_v5_check
NPY_DIR: /kaggle/input/datasets/mizushimatoshihiko/npy-files


In [2]:
# ============================================================
# 1. Utilities
# ============================================================

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def load_npy(filename):
    path = NPY_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing npy file: {path}")
    return path

def validate_proba(name, arr, n_rows, n_classes):
    assert arr.shape == (n_rows, n_classes), (name, arr.shape)
    assert np.isfinite(arr).all(), name
    assert np.allclose(arr.sum(axis=1), 1.0, atol=1e-4), name
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

def normalize_rows(p, eps=1e-12):
    p = np.clip(p, eps, None)
    return p / p.sum(axis=1, keepdims=True)

def softmax_np(x, axis=1):
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def proba_to_pred(p):
    return np.argmax(p, axis=1)

def flatten_proba(p):
    return np.asarray(p, dtype=float).reshape(-1)

def corr_pearson(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def corr_spearman(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(spearmanr(a, b).correlation)

def class_recalls(y_true, y_pred, class_names):
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out

def evaluate_proba(y_true, p, class_names):
    pred = proba_to_pred(p)
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    res.update(class_recalls(y_true, pred, class_names))
    return res

def rank_normalize_matrix(p):
    out = np.zeros_like(p, dtype=np.float64)
    n = p.shape[0]
    for j in range(p.shape[1]):
        r = pd.Series(p[:, j]).rank(method="average").to_numpy()
        out[:, j] = (r - 1) / max(n - 1, 1)
    return normalize_rows(out)

def logit_transform(p, eps=1e-15):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def build_meta_features(keys, proba_dict, mode):
    mats = []
    for k in keys:
        p = proba_dict[k]
        if mode == "raw":
            mats.append(p)
        elif mode == "rank":
            mats.append(rank_normalize_matrix(p))
        elif mode == "logit":
            mats.append(logit_transform(p))
        elif mode == "raw_logit":
            mats += [p, logit_transform(p)]
        elif mode == "raw_rank_logit":
            mats += [p, rank_normalize_matrix(p), logit_transform(p)]
        else:
            raise ValueError(mode)
    return np.concatenate(mats, axis=1)

def average_proba(keys, oof_dict, pred_dict=None):
    oof_avg = normalize_rows(np.mean([oof_dict[k] for k in keys], axis=0))
    pred_avg = None
    if pred_dict is not None:
        pred_avg = normalize_rows(np.mean([pred_dict[k] for k in keys], axis=0))
    return oof_avg, pred_avg

def hill_climb_weights(y_true, probas, steps=(0.1, 0.05, 0.02, 0.01, 0.005, 0.002), max_iter=300):
    n = len(probas)
    w = np.ones(n, dtype=float) / n
    cur_score = balanced_accuracy_score(y_true, normalize_rows(sum(w[i] * probas[i] for i in range(n))).argmax(axis=1))
    hist = [{"iter": 0, "score": float(cur_score), "weights": w.copy().tolist()}]
    for step in steps:
        improved = True
        it = 0
        while improved and it < max_iter:
            improved = False
            it += 1
            best_score, best_w = cur_score, w.copy()
            for i in range(n):
                for direction in [-1, 1]:
                    cand_w = w.copy()
                    cand_w[i] += direction * step
                    cand_w = np.clip(cand_w, 0.0, None)
                    if cand_w.sum() <= 0:
                        continue
                    cand_w /= cand_w.sum()
                    cand = normalize_rows(sum(cand_w[j] * probas[j] for j in range(n)))
                    score = balanced_accuracy_score(y_true, cand.argmax(axis=1))
                    if score > best_score + 1e-15:
                        best_score, best_w = score, cand_w.copy()
            if best_score > cur_score + 1e-15:
                cur_score, w, improved = best_score, best_w, True
                hist.append({"iter": len(hist), "score": float(cur_score), "weights": w.copy().tolist()})
    final = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    return w, final, float(cur_score), hist

def nm_optimize_weights(y_true, probas, maxiter=1500):
    n = len(probas)
    def unpack(theta):
        e = np.exp(theta - np.max(theta))
        return e / e.sum()
    def objective(theta):
        w = unpack(theta)
        p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
        return -balanced_accuracy_score(y_true, p.argmax(axis=1))
    res = minimize(objective, np.zeros(n), method="Nelder-Mead",
                   options={"maxiter": maxiter, "xatol": 1e-7, "fatol": 1e-10})
    w = unpack(res.x)
    p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    score = balanced_accuracy_score(y_true, p.argmax(axis=1))
    return w, p, float(score), res

def onehot_y(y, n_classes):
    out = np.zeros((len(y), n_classes), dtype=np.float64)
    out[np.arange(len(y)), y] = 1.0
    return out

In [3]:
# ============================================================
# 2. Load data and OOF/pred
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH, NPY_DIR]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)
assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof, pred, resolved_specs = {}, {}, []
for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = load_npy(spec["oof"])
    pred_path = load_npy(spec["pred"])
    oof_arr = np.load(oof_path).astype(np.float32)
    pred_arr = np.load(pred_path).astype(np.float32)
    validate_proba(f"{key} oof", oof_arr, len(train), n_classes)
    validate_proba(f"{key} pred", pred_arr, len(test), n_classes)
    oof[key] = oof_arr
    pred[key] = pred_arr
    row = dict(spec)
    row["resolved_oof_path"] = str(oof_path)
    row["resolved_pred_path"] = str(pred_path)
    row["oof_shape"] = list(oof_arr.shape)
    row["pred_shape"] = list(pred_arr.shape)
    resolved_specs.append(row)

model_keys = [s["key"] for s in MODEL_SPECS]
display(pd.DataFrame(resolved_specs))

,key,exp_id,family,role,oof,pred,cv,public_lb,resolved_oof_path,resolved_pred_path,oof_shape,pred_shape
0,014_blend_bias,exp_20260605_014_bias_search_on_013_best_blend,Blend,old_blend_bias_reference,oof_exp_20260605_014_bias_search_on_013_best_b...,pred_exp_20260605_014_bias_search_on_013_best_...,0.966009,0.96703,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
1,015_realmlp_seed42,exp_20260605_015_shared001_realmlp_as_is,RealMLP,stable_single_realmlp_backup,oof_exp_20260605_015_shared001_realmlp_as_is_p...,pred_exp_20260605_015_shared001_realmlp_as_is_...,0.968169,0.96977,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
2,017_realmlp_bias,exp_20260606_017_bias_search_on_015_realmlp,RealMLP,previous_best_realmlp_bias_backup,oof_exp_20260606_017_bias_search_on_015_realml...,pred_exp_20260606_017_bias_search_on_015_realm...,0.968302,0.96985,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
3,018_xgb_shared003,exp_20260606_018_shared003_xgb_as_is,XGBoost,effective_blend_material,oof_exp_20260606_018_shared003_xgb_as_is_proba...,pred_exp_20260606_018_shared003_xgb_as_is_prob...,0.965208,0.96578,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
4,019_blend_best,exp_20260607_019_blend_add018_xgb_check,Blend,public_confirmed_current_best,oof_exp_20260607_019_blend_add018_xgb_check_be...,pred_exp_20260607_019_blend_add018_xgb_check_b...,0.968872,0.97000,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
5,020_blend_bias,exp_20260607_020_bias_search_on_019_best_blend,Blend,cv_trusted_attack_reference,oof_exp_20260607_020_bias_search_on_019_best_b...,pred_exp_20260607_020_bias_search_on_019_best_...,0.969224,0.96969,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
6,024_seed_ensemble_blend,exp_20260608_024_seed_ensemble_and_blend_check,Blend,cv_trusted_slot,oof_exp_20260608_024_seed_ensemble_and_blend_c...,pred_exp_20260608_024_seed_ensemble_and_blend_...,0.969480,0.96958,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
7,026_realmlp_v5,exp_20260608_026_realmlp_v5_as_is,RealMLP,realmlp_v5_single_model_candidate,oof_exp_20260608_026_realmlp_v5_as_is_proba.npy,pred_exp_20260608_026_realmlp_v5_as_is_proba.npy,0.969039,0.96979,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"


In [4]:
# ============================================================
# 3. Individual scores and pairwise diagnostics
# ============================================================

rows = []
for spec in resolved_specs:
    key = spec["key"]
    p = oof[key]
    y_pred = proba_to_pred(p)
    score = balanced_accuracy_score(y, y_pred)
    row = {
        "key": key,
        "exp_id": spec["exp_id"],
        "family": spec["family"],
        "role": spec["role"],
        "cv_from_memo": spec.get("cv", np.nan),
        "public_lb": spec.get("public_lb", np.nan),
        "recomputed_oof_cv": float(score),
        "delta_recomputed_minus_memo": float(score - spec.get("cv", score)),
    }
    row.update(class_recalls(y, y_pred, class_names))
    rows.append(row)

individual_df = pd.DataFrame(rows).sort_values("recomputed_oof_cv", ascending=False)
display(individual_df)
individual_df.to_csv(OUTDIR / "individual_scores.csv", index=False)

pred_labels = {k: proba_to_pred(oof[k]) for k in model_keys}
wrong_flags = {k: pred_labels[k] != y for k in model_keys}
pair_rows = []

for a, b in combinations(model_keys, 2):
    pa, pb = oof[a], oof[b]
    ya, yb = pred_labels[a], pred_labels[b]
    wrong_a, wrong_b = wrong_flags[a], wrong_flags[b]
    both = wrong_a & wrong_b
    either = wrong_a | wrong_b
    row = {
        "model_a": a,
        "model_b": b,
        "pearson_flat_proba": corr_pearson(pa, pb),
        "spearman_flat_proba": corr_spearman(pa, pb),
        "argmax_agreement": float((ya == yb).mean()),
        "argmax_disagreement": float((ya != yb).mean()),
        "wrong_rate_a": float(wrong_a.mean()),
        "wrong_rate_b": float(wrong_b.mean()),
        "both_wrong_rate": float(both.mean()),
        "either_wrong_rate": float(either.mean()),
        "error_jaccard": float(both.sum() / max(either.sum(), 1)),
        "a_wrong_b_correct_rate": float((wrong_a & ~wrong_b).mean()),
        "a_correct_b_wrong_rate": float((~wrong_a & wrong_b).mean()),
    }
    for ci, cls in enumerate(class_names):
        row[f"pearson_proba_{cls}"] = corr_pearson(pa[:, ci], pb[:, ci])
        row[f"spearman_proba_{cls}"] = corr_spearman(pa[:, ci], pb[:, ci])
    pair_rows.append(row)

pairwise_df = pd.DataFrame(pair_rows).sort_values("pearson_flat_proba")
display(pairwise_df)
pairwise_df.to_csv(OUTDIR / "pairwise_oof_correlation.csv", index=False)

pearson_flat = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
spearman_flat = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
argmax_agreement = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
error_jaccard = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)

for a in model_keys:
    for b in model_keys:
        pearson_flat.loc[a, b] = corr_pearson(oof[a], oof[b])
        spearman_flat.loc[a, b] = corr_spearman(oof[a], oof[b])
        argmax_agreement.loc[a, b] = float((pred_labels[a] == pred_labels[b]).mean())
        both = wrong_flags[a] & wrong_flags[b]
        either = wrong_flags[a] | wrong_flags[b]
        error_jaccard.loc[a, b] = float(both.sum() / max(either.sum(), 1))

pearson_flat.to_csv(OUTDIR / "corr_matrix_pearson_flat_proba.csv")
spearman_flat.to_csv(OUTDIR / "corr_matrix_spearman_flat_proba.csv")
argmax_agreement.to_csv(OUTDIR / "matrix_argmax_agreement.csv")
error_jaccard.to_csv(OUTDIR / "matrix_error_jaccard.csv")
display(pearson_flat)
display(spearman_flat)
display(error_jaccard)

,key,exp_id,family,role,cv_from_memo,public_lb,recomputed_oof_cv,delta_recomputed_minus_memo,recall_GALAXY,recall_QSO,recall_STAR
6,024_seed_ensemble_blend,exp_20260608_024_seed_ensemble_and_blend_check,Blend,cv_trusted_slot,0.969480,0.96958,0.969480,0.0,0.961956,0.976174,0.970311
5,020_blend_bias,exp_20260607_020_bias_search_on_019_best_blend,Blend,cv_trusted_attack_reference,0.969224,0.96969,0.969224,0.0,0.961137,0.976200,0.970335
7,026_realmlp_v5,exp_20260608_026_realmlp_v5_as_is,RealMLP,realmlp_v5_single_model_candidate,0.969039,0.96979,0.969039,0.0,0.959505,0.976431,0.971181
4,019_blend_best,exp_20260607_019_blend_add018_xgb_check,Blend,public_confirmed_current_best,0.968872,0.97000,0.968872,0.0,0.965479,0.974441,0.966696
2,017_realmlp_bias,exp_20260606_017_bias_search_on_015_realmlp,RealMLP,previous_best_realmlp_bias_backup,0.968302,0.96985,0.968302,0.0,0.959889,0.975867,0.969150
1,015_realmlp_seed42,exp_20260605_015_shared001_realmlp_as_is,RealMLP,stable_single_realmlp_backup,0.968169,0.96977,0.968169,0.0,0.962234,0.976098,0.966177
0,014_blend_bias,exp_20260605_014_bias_search_on_013_best_blend,Blend,old_blend_bias_reference,0.966009,0.96703,0.966009,0.0,0.968062,0.970617,0.959347
3,018_xgb_shared003,exp_20260606_018_shared003_xgb_as_is,XGBoost,effective_blend_material,0.965208,0.96578,0.965208,0.0,0.966870,0.972043,0.956711


,model_a,model_b,pearson_flat_proba,spearman_flat_proba,argmax_agreement,argmax_disagreement,wrong_rate_a,wrong_rate_b,both_wrong_rate,either_wrong_rate,error_jaccard,a_wrong_b_correct_rate,a_correct_b_wrong_rate,pearson_proba_GALAXY,spearman_proba_GALAXY,pearson_proba_QSO,spearman_proba_QSO,pearson_proba_STAR,spearman_proba_STAR
13,017_realmlp_bias,018_xgb_shared003,0.990670,0.908650,0.981524,0.018476,0.035542,0.033536,0.025439,0.043639,0.582933,0.010103,0.008097,0.987587,0.939083,0.994756,0.857918,0.980882,0.780807
8,015_realmlp_seed42,018_xgb_shared003,0.991146,0.918772,0.982243,0.017757,0.034388,0.033536,0.025227,0.042697,0.590848,0.009161,0.008309,0.988216,0.939921,0.994776,0.854090,0.981740,0.778653
21,018_xgb_shared003,026_realmlp_v5,0.991382,0.794941,0.982709,0.017291,0.033536,0.035388,0.025946,0.042978,0.603716,0.007590,0.009441,0.989240,0.920986,0.995267,0.812467,0.983883,0.797986
6,014_blend_bias,026_realmlp_v5,0.993470,0.787477,0.985277,0.014723,0.032668,0.035388,0.026816,0.041240,0.650231,0.005853,0.008572,0.992409,0.928704,0.996069,0.772104,0.988981,0.812959
1,014_blend_bias,017_realmlp_bias,0.993651,0.904324,0.985206,0.014794,0.032668,0.035542,0.026857,0.041353,0.649466,0.005811,0.008685,0.991905,0.941093,0.995901,0.851459,0.987737,0.773884
2,014_blend_bias,018_xgb_shared003,0.993720,0.976021,0.984379,0.015621,0.032668,0.033536,0.025427,0.040778,0.623540,0.007242,0.008110,0.991644,0.967257,0.996035,0.894090,0.986881,0.925413
0,014_blend_bias,015_realmlp_seed42,0.994167,0.916308,0.985937,0.014063,0.032668,0.034388,0.026648,0.040409,0.659451,0.006021,0.007741,0.992597,0.941779,0.995879,0.854700,0.988644,0.771301
20,018_xgb_shared003,024_seed_ensemble_blend,0.994640,0.938731,0.985941,0.014059,0.033536,0.033962,0.026821,0.040677,0.659357,0.006715,0.007141,0.992833,0.960224,0.996898,0.893590,0.989190,0.811215
19,018_xgb_shared003,020_blend_bias,0.995772,0.933916,0.987474,0.012526,0.033536,0.034489,0.027838,0.040187,0.692699,0.005698,0.006651,0.994357,0.962761,0.997602,0.897744,0.991549,0.818295
5,014_blend_bias,024_seed_ensemble_blend,0.996230,0.937715,0.988296,0.011704,0.032668,0.033962,0.027590,0.039041,0.706699,0.005078,0.006372,0.995198,0.965744,0.997494,0.888994,0.992934,0.813909


,014_blend_bias,015_realmlp_seed42,017_realmlp_bias,018_xgb_shared003,019_blend_best,020_blend_bias,024_seed_ensemble_blend,026_realmlp_v5
014_blend_bias,1.000000,0.994167,0.993651,0.993720,0.997309,0.996506,0.996230,0.993470
015_realmlp_seed42,0.994167,1.000000,0.999895,0.991146,0.998530,0.998570,0.998846,0.997331
017_realmlp_bias,0.993651,0.999895,1.000000,0.990670,0.998337,0.998634,0.998863,0.997496
018_xgb_shared003,0.993720,0.991146,0.990670,1.000000,0.996517,0.995772,0.994640,0.991382
019_blend_best,0.997309,0.998530,0.998337,0.996517,1.000000,0.999766,0.999481,0.997262
020_blend_bias,0.996506,0.998570,0.998634,0.995772,0.999766,1.000000,0.999654,0.997709
024_seed_ensemble_blend,0.996230,0.998846,0.998863,0.994640,0.999481,0.999654,1.000000,0.998117
026_realmlp_v5,0.993470,0.997331,0.997496,0.991382,0.997262,0.997709,0.998117,1.000000


,014_blend_bias,015_realmlp_seed42,017_realmlp_bias,018_xgb_shared003,019_blend_best,020_blend_bias,024_seed_ensemble_blend,026_realmlp_v5
014_blend_bias,1.000000,0.916308,0.904324,0.976021,0.936711,0.930534,0.937715,0.787477
015_realmlp_seed42,0.916308,1.000000,0.997811,0.918772,0.992035,0.989531,0.962640,0.816937
017_realmlp_bias,0.904324,0.997811,1.000000,0.908650,0.990462,0.990901,0.961756,0.834952
018_xgb_shared003,0.976021,0.918772,0.908650,1.000000,0.937942,0.933916,0.938731,0.794941
019_blend_best,0.936711,0.992035,0.990462,0.937942,1.000000,0.997020,0.971993,0.821950
020_blend_bias,0.930534,0.989531,0.990901,0.933916,0.997020,1.000000,0.974434,0.847534
024_seed_ensemble_blend,0.937715,0.962640,0.961756,0.938731,0.971993,0.974434,1.000000,0.844631
026_realmlp_v5,0.787477,0.816937,0.834952,0.794941,0.821950,0.847534,0.844631,1.000000


,014_blend_bias,015_realmlp_seed42,017_realmlp_bias,018_xgb_shared003,019_blend_best,020_blend_bias,024_seed_ensemble_blend,026_realmlp_v5
014_blend_bias,1.000000,0.659451,0.649466,0.623540,0.742478,0.710850,0.706699,0.650231
015_realmlp_seed42,0.659451,1.000000,0.941244,0.590848,0.814570,0.827230,0.843588,0.774435
017_realmlp_bias,0.649466,0.941244,1.000000,0.582933,0.798627,0.825950,0.842001,0.781795
018_xgb_shared003,0.623540,0.590848,0.582933,1.000000,0.714710,0.692699,0.659357,0.603716
019_blend_best,0.742478,0.814570,0.798627,0.714710,1.000000,0.894809,0.876980,0.764116
020_blend_bias,0.710850,0.827230,0.825950,0.692699,0.894809,1.000000,0.900914,0.790158
024_seed_ensemble_blend,0.706699,0.843588,0.842001,0.659357,0.876980,0.900914,1.000000,0.807956
026_realmlp_v5,0.650231,0.774435,0.781795,0.603716,0.764116,0.790158,0.807956,1.000000


In [5]:
# ============================================================
# 4. Blend diagnostics
# ============================================================

blend_rows = {}

def record_blend(set_name, method, keys, oof_p, test_p=None, weights=None, extra=None):
    ev = evaluate_proba(y, oof_p, class_names)
    row = {
        "set_name": set_name,
        "method": method,
        "keys": ",".join(keys),
        "n_models": len(keys),
        "balanced_accuracy": ev["balanced_accuracy"],
        "weights_json": json.dumps({k: float(w) for k, w in zip(keys, weights)}, ensure_ascii=False) if weights is not None else None,
    }
    row.update({k: v for k, v in ev.items() if k != "balanced_accuracy"})
    if extra:
        row.update(extra)
    blend_rows[(set_name, method)] = row

for set_name, keys in BLEND_SETS.items():
    print(f"===== {set_name}: {keys} =====")
    probas = [oof[k] for k in keys]
    pred_probas = [pred[k] for k in keys]

    avg_oof, avg_pred = average_proba(keys, oof, pred)
    record_blend(set_name, "avg", keys, avg_oof, avg_pred, weights=np.ones(len(keys)) / len(keys))

    if len(keys) >= 2:
        hc_w, hc_oof, _, hc_hist = hill_climb_weights(y, probas)
        hc_pred = normalize_rows(sum(hc_w[i] * pred_probas[i] for i in range(len(keys))))
        record_blend(set_name, "hc_nonnegative_raw", keys, hc_oof, hc_pred, weights=hc_w, extra={"hc_history_len": len(hc_hist)})

        nm_w, nm_oof, _, nm_res = nm_optimize_weights(y, probas)
        nm_pred = normalize_rows(sum(nm_w[i] * pred_probas[i] for i in range(len(keys))))
        record_blend(set_name, "nm_softmax_raw", keys, nm_oof, nm_pred, weights=nm_w, extra={"nm_success": bool(nm_res.success), "nm_message": str(nm_res.message)})

    # Diagnostic only, not nested OOF.
    for mode in ["raw", "logit", "raw_logit", "raw_rank_logit"]:
        X_meta = build_meta_features(keys, oof, mode)
        X_test_meta = build_meta_features(keys, pred, mode)
        try:
            lr = LogisticRegression(C=0.05, penalty="l2", solver="lbfgs", max_iter=2000, random_state=SEED, n_jobs=-1)
            lr.fit(X_meta, y)
            record_blend(set_name, f"logreg_{mode}", keys, lr.predict_proba(X_meta), lr.predict_proba(X_test_meta), extra={"C": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] logreg failed: {set_name} {mode}: {e}")
        try:
            rc = RidgeClassifier(alpha=10.0, random_state=SEED)
            rc.fit(X_meta, y)
            record_blend(set_name, f"ridgeclf_{mode}", keys, softmax_np(rc.decision_function(X_meta)), softmax_np(rc.decision_function(X_test_meta)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridgeclf failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            rr = Ridge(alpha=10.0, random_state=SEED)
            rr.fit(X_meta, y_oh)
            record_blend(set_name, f"ridge_reg_{mode}", keys, normalize_rows(np.clip(rr.predict(X_meta), 1e-12, None)), normalize_rows(np.clip(rr.predict(X_test_meta), 1e-12, None)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridge_reg failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            oof_cols, test_cols = [], []
            for c in range(n_classes):
                en = ElasticNet(alpha=0.0005, l1_ratio=0.05, random_state=SEED, max_iter=2000)
                en.fit(X_meta, y_oh[:, c])
                oof_cols.append(en.predict(X_meta))
                test_cols.append(en.predict(X_test_meta))
            en_oof = normalize_rows(np.clip(np.vstack(oof_cols).T, 1e-12, None))
            en_pred = normalize_rows(np.clip(np.vstack(test_cols).T, 1e-12, None))
            record_blend(set_name, f"elasticnet_reg_{mode}", keys, en_oof, en_pred, extra={"alpha": 0.0005, "l1_ratio": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] elasticnet_reg failed: {set_name} {mode}: {e}")

blend_df = pd.DataFrame(list(blend_rows.values())).sort_values("balanced_accuracy", ascending=False)
display(blend_df.head(180))
blend_df.to_csv(OUTDIR / "blend_diagnostics.csv", index=False)

===== A_019_only: ['019_blend_best'] =====
===== B_024_only: ['024_seed_ensemble_blend'] =====
===== C_020_only: ['020_blend_bias'] =====
===== D_026_only: ['026_realmlp_v5'] =====
[WARN] logreg failed: D_026_only logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: D_026_only logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] ridge_reg failed: D_026_only logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] elasticnet_reg failed: D_026_only logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] logreg failed: D_026_only raw_logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: D_026_only raw_logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] ridge_reg failed: D_026_only raw_logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] elasticnet_reg fail

,set_name,method,keys,n_models,balanced_accuracy,weights_json,recall_GALAXY,recall_QSO,recall_STAR,C,risk,alpha,l1_ratio,hc_history_len,nm_success,nm_message
111,M_020_024_026,hc_nonnegative_raw,"020_blend_bias,024_seed_ensemble_blend,026_rea...",3,0.969543,"{""020_blend_bias"": 0.04898792747788589, ""024_s...",0.961479,0.976439,0.970710,NaN,NaN,NaN,NaN,11.0,NaN,NaN
69,G_024_026,hc_nonnegative_raw,"024_seed_ensemble_blend,026_realmlp_v5",2,0.969526,"{""024_seed_ensemble_blend"": 0.7255189856779073...",0.961505,0.976388,0.970686,NaN,NaN,NaN,NaN,9.0,NaN,NaN
139,Q_014_018_024_026,hc_nonnegative_raw,"014_blend_bias,018_xgb_shared003,024_seed_ense...",4,0.969510,"{""014_blend_bias"": 0.0, ""018_xgb_shared003"": 0...",0.962634,0.976081,0.969815,NaN,NaN,NaN,NaN,15.0,NaN,NaN
146,R_014_018_019_024_026,hc_nonnegative_raw,"014_blend_bias,018_xgb_shared003,019_blend_bes...",5,0.969493,"{""014_blend_bias"": 0.0, ""018_xgb_shared003"": 0...",0.962843,0.975918,0.969719,NaN,NaN,NaN,NaN,11.0,NaN,NaN
118,N_019_020_024_026,hc_nonnegative_raw,"019_blend_best,020_blend_bias,024_seed_ensembl...",4,0.969482,"{""019_blend_best"": 0.180116094643635, ""020_ble...",0.961974,0.976149,0.970323,NaN,NaN,NaN,NaN,10.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7,A_019_only,ridge_reg_logit,019_blend_best,1,0.959464,None,0.978934,0.962584,0.936874,NaN,in_sample_meta_screening,10.0000,NaN,NaN,NaN,NaN
6,A_019_only,ridgeclf_logit,019_blend_best,1,0.959464,None,0.978934,0.962584,0.936874,NaN,in_sample_meta_screening,10.0000,NaN,NaN,NaN,NaN
42,C_020_only,elasticnet_reg_logit,020_blend_bias,1,0.959309,None,0.978960,0.962345,0.936621,NaN,in_sample_meta_screening,0.0005,0.05,NaN,NaN,NaN
40,C_020_only,ridgeclf_logit,020_blend_bias,1,0.959300,None,0.978982,0.962311,0.936608,NaN,in_sample_meta_screening,10.0000,NaN,NaN,NaN,NaN


In [6]:
# ============================================================
# 5. Save best safe blend
# ============================================================

safe_methods = ["avg", "hc_nonnegative_raw", "nm_softmax_raw"]
safe_blend_df = blend_df[blend_df["method"].isin(safe_methods)].copy()
safe_blend_df = safe_blend_df.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
safe_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics.csv", index=False)

best_row = safe_blend_df.iloc[0].to_dict()
best_set_name = best_row["set_name"]
best_method = best_row["method"]
best_keys = best_row["keys"].split(",")

print("Best safe blend:")
print(json.dumps(best_row, indent=2, ensure_ascii=False))

if best_method == "avg":
    best_oof_proba, best_pred_proba = average_proba(best_keys, oof, pred)
else:
    weights_json = json.loads(best_row["weights_json"])
    w = np.array([weights_json[k] for k in best_keys], dtype=float)
    best_oof_proba = normalize_rows(sum(w[i] * oof[best_keys[i]] for i in range(len(best_keys))))
    best_pred_proba = normalize_rows(sum(w[i] * pred[best_keys[i]] for i in range(len(best_keys))))

validate_proba("best_oof_proba", best_oof_proba, len(train), n_classes)
validate_proba("best_pred_proba", best_pred_proba, len(test), n_classes)

best_oof_score = balanced_accuracy_score(y, best_oof_proba.argmax(axis=1))
print("best_oof_score:", best_oof_score)

best_oof_npy = OUTDIR / "oof_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy"
best_pred_npy = OUTDIR / "pred_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy"
np.save(best_oof_npy, best_oof_proba.astype(np.float32))
np.save(best_pred_npy, best_pred_proba.astype(np.float32))

best_labels = le.inverse_transform(best_pred_proba.argmax(axis=1))
submission = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: best_labels})
assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

best_submission_path = OUTDIR / "submission_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend.csv"
submission.to_csv(best_submission_path, index=False)

best_info = {
    "exp_id": EXP_ID,
    "best_set_name": best_set_name,
    "best_method": best_method,
    "best_keys": best_keys,
    "cv_score": float(best_oof_score),
    "weights_json": best_row.get("weights_json"),
    "oof_path": str(best_oof_npy),
    "pred_path": str(best_pred_npy),
    "submission_path": str(best_submission_path),
    "row": best_row,
    "note": "Best safe blend excludes in-sample meta screening methods.",
}
save_json(best_info, OUTDIR / "best_blend_info.json")

saved_submission_df = pd.DataFrame([{
    "set_name": best_set_name,
    "method": best_method,
    "balanced_accuracy": float(best_oof_score),
    "submission": best_submission_path.name,
    "oof_proba": best_oof_npy.name,
    "pred_proba": best_pred_npy.name,
}])
display(saved_submission_df)
saved_submission_df.to_csv(OUTDIR / "saved_submission_candidates.csv", index=False)

Best safe blend:
{
  "set_name": "M_020_024_026",
  "method": "hc_nonnegative_raw",
  "keys": "020_blend_bias,024_seed_ensemble_blend,026_realmlp_v5",
  "n_models": 3,
  "balanced_accuracy": 0.9695425457477898,
  "weights_json": "{\"020_blend_bias\": 0.04898792747788589, \"024_seed_ensemble_blend\": 0.6890973266133963, \"026_realmlp_v5\": 0.2619147459087178}",
  "recall_GALAXY": 0.9614787538412631,
  "recall_QSO": 0.976439053123106,
  "recall_STAR": 0.970709830279,
  "C": NaN,
  "risk": NaN,
  "alpha": NaN,
  "l1_ratio": NaN,
  "hc_history_len": 11.0,
  "nm_success": NaN,
  "nm_message": NaN
}
best_oof_score: 0.9695425457477898


,set_name,method,balanced_accuracy,submission,oof_proba,pred_proba
0,M_020_024_026,hc_nonnegative_raw,0.969543,submission_exp_20260608_027_blend_add026_realm...,oof_exp_20260608_027_blend_add026_realmlp_v5_c...,pred_exp_20260608_027_blend_add026_realmlp_v5_...


In [7]:
# ============================================================
# 6. Save summary / memo
# ============================================================

judgement = {
    "reference_scores": {
        "019_cv": 0.968872315017554,
        "019_public_lb": 0.97000,
        "024_cv": 0.9694803109983217,
        "024_public_lb": 0.96958,
        "020_cv": 0.9692240443617589,
        "020_public_lb": 0.96969,
        "026_cv": 0.9690389777981231,
        "026_public_lb": 0.96979,
        "017_cv": 0.9683022653955233,
        "017_public_lb": 0.96985,
        "015_cv": 0.9681693449222352,
        "015_public_lb": 0.96977,
        "018_cv": 0.965207986418628,
        "018_public_lb": 0.96578,
    },
    "individual_best": {
        "key": individual_df.iloc[0]["key"],
        "cv": float(individual_df.iloc[0]["recomputed_oof_cv"]),
        "public_lb": float(individual_df.iloc[0]["public_lb"]),
    },
    "best_safe_blend": best_info,
    "main_questions": {
        "does_026_add_to_019": "Check F/P/R/S safe methods and 026 weight.",
        "does_026_add_to_024": "Check G/K/N/Q/R/S safe methods and 026 weight.",
        "should_bias_search_next": "Only if 027 best safe blend improves meaningfully and has stable weights.",
        "next_family": "If 026 is high-correlation or weak in blend, move to CatBoost v3 / OVR XGB / TabM.",
    },
}
save_json(judgement, OUTDIR / "role_judgement.json")

summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "status": "completed",
    "purpose": "Blend/correlation check after adding 026 RealMLP v5 as-is to current candidates",
    "npy_dir": str(NPY_DIR),
    "model_keys": model_keys,
    "class_names": class_names,
    "resolved_specs": resolved_specs,
    "individual_scores": individual_df.to_dict(orient="records"),
    "pairwise_oof_correlation": pairwise_df.to_dict(orient="records"),
    "blend_diagnostics": blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics": safe_blend_df.to_dict(orient="records"),
    "best_blend_info": best_info,
    "saved_submission_candidates": saved_submission_df.to_dict(orient="records"),
    "role_judgement": judgement,
}
save_json(summary, OUTDIR / "blend_add026_realmlp_v5_summary.json")

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "Blend check after adding 026 RealMLP v5",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "created_at": "2026-06-08",
    },
    "objective": {
        "summary": (
            "Load OOF/pred probabilities from npy-files, add 026 RealMLP v5 as-is, "
            "and decide whether this separate RealMLP recipe adds complementary signal to 019/024/020/017/015/018."
        ),
        "success_criteria": [
            "load 014/015/017/018/019/020/024/026 OOF and pred npy files",
            "recompute individual balanced accuracy",
            "compute pairwise OOF correlations",
            "evaluate add026 blend sets",
            "separate safe blends from in-sample meta screening",
            "save best safe blend OOF/pred npy",
            "save best safe blend submission",
        ],
    },
    "inputs": {"npy_dir": str(NPY_DIR), "models": resolved_specs, "blend_sets": BLEND_SETS},
    "outputs": {
        "individual_scores": "individual_scores.csv",
        "pairwise_oof_correlation": "pairwise_oof_correlation.csv",
        "blend_diagnostics": "blend_diagnostics.csv",
        "safe_blend_diagnostics": "safe_blend_diagnostics.csv",
        "best_blend_info": "best_blend_info.json",
        "best_oof_proba": best_oof_npy.name,
        "best_pred_proba": best_pred_npy.name,
        "best_submission": best_submission_path.name,
        "summary": "blend_add026_realmlp_v5_summary.json",
    },
    "judgement": {
        "status": "completed_pending_manual_review",
        "initial_view": (
            "026 is a strong RealMLP single model and a separate recipe from 015. "
            "Its value depends on whether it keeps nonzero weight in safe blends with 019/024/020."
        ),
        "next_action": [
            "Review individual_scores.csv",
            "Review pairwise_oof_correlation.csv, especially 026 vs 019/024/017/015",
            "Review safe rows in blend_diagnostics.csv",
            "Submit best safe blend only if it improves over 019/024 enough",
            "If best safe blend improves, consider bias search on 027 best",
            "If 026 gets near-zero weight, keep it as a single backup and move to CatBoost v3",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)

Saved files:
 - best_blend_info.json
 - blend_add026_realmlp_v5_summary.json
 - blend_diagnostics.csv
 - corr_matrix_pearson_flat_proba.csv
 - corr_matrix_spearman_flat_proba.csv
 - individual_scores.csv
 - matrix_argmax_agreement.csv
 - matrix_error_jaccard.csv
 - memo.yml
 - oof_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy
 - pairwise_oof_correlation.csv
 - pred_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy
 - role_judgement.json
 - safe_blend_diagnostics.csv
 - saved_submission_candidates.csv
 - submission_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend.csv
